In [8]:
import json
from pathlib import Path
import pandas as pd

# 1. Carrega o arquivo dados.json
json_path = Path("dados.json")
if not json_path.exists():
    json_path = Path("/workspaces/Fundamentos/notebooks/dados.json")

conteudo = json.loads(json_path.read_text(encoding="utf-8"))
if isinstance(conteudo, dict):
    conteudo = [conteudo]

linhas = []
for item in conteudo:
    nome = item.get("nome_completo", item.get("nome", ""))
    id_lattes = item.get("id_lattes", item.get("id", ""))

    artigos = item.get(
        "artigos_publicados",
        item.get("producao_bibliografica", {}).get("artigos_publicados", []),
    )

    if isinstance(artigos, list):
        for artigo in artigos:
            if isinstance(artigo, dict):
                linhas.append(
                    {
                        "nome_completo": nome,
                        "id_lattes": id_lattes,
                        "titulo_artigo": artigo.get("titulo", ""),
                        "ano": artigo.get("ano", ""),
                    }
                )
    elif isinstance(artigos, dict):
        linhas.append(
            {
                "nome_completo": nome,
                "id_lattes": id_lattes,
                "titulo_artigo": artigos.get("titulo", ""),
                "ano": artigos.get("ano", ""),
            }
        )

df = pd.DataFrame(linhas)

# 2. Tratamento de tipos (Int64 para id_lattes e ano)
for col in ["id_lattes", "ano"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

# 3. Higienização de strings
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype(str).str.strip()

# 4. Verificação dos tipos
print("--- Tipos de Dados Convertidos ---")
print(df.dtypes)
print("\n--- Primeiras Linhas ---")
print(df.head())

# 5. Exportação otimizada para Parquet
df.to_parquet("artigos.parquet", index=False)
print("\n✅ Arquivo 'artigos.parquet' salvo com sucesso!")

--- Tipos de Dados Convertidos ---
nome_completo      str
id_lattes        Int64
titulo_artigo      str
ano              Int64
dtype: object

--- Primeiras Linhas ---
  nome_completo         id_lattes titulo_artigo   ano
0     Ana Silva  1234567890123456      Artigo A  2023
1     Ana Silva  1234567890123456      Artigo B  2024

✅ Arquivo 'artigos.parquet' salvo com sucesso!
